# nb06_reasoning_report : R1 추론 로직 검증 · 판독 보조 리포트 (M4)

- **가속기 : None (CPU)** · **인터넷 : 불필요**
- **입력** : Dataset `hms-src`(최신), **Dataset `hms_processed`**(사례 그림용 eeg50·spec10), 노트북 `nb03_profile_lgbm` Output(profile_features.csv), 노트북 `nb08_pack` Output(E4·E5·P1 OOF, e8_recipe.json)
- **출력** : `/kaggle/working/reasoning/` (구간별 판정·근거, 임계값, 클래스별 표, 사례 후보, 요약)

| 섹션 | 내용 | 코드북 |
|---|---|---|
| 0 | 설정 | - |
| 1 | E8 확률 재구성 (0.85·mean(E4, E5) + 0.15·P1v2) | E8 |
| 2 | **규칙 적용** : fold k 임계값은 나머지 fold 고득표 구간에서 학습 → fold k에 적용 | 5장 (임계값 누수 방지) |
| 3 | **평가** : 고득표 근거 일치율(목표 ≥ 80%), 불일치 플래그 샘플 오답률(일치 대비 ≥ 2배) | 성능 목표 (추론 로직) |
| 4 | 판독 보조 리포트 + **사례 리포트 8건**(문구 + 파형·스펙트로그램 그림, case_reports.md) | 목표 2, M5 |
| 5 | 저장 | results.csv R1 |

**근거 지표 (H5 지지 지표만, 20차 수정 100)**
| 판정 | 주장 | 근거 지표 (각 2개 중 1개 이상 충족 시 지지) |
|---|---|---|
| LPD | 편측성 높음 (+ 주기성 : 근거 미확정) | abs_asym_total, abs_asym_delta |
| GPD | 편측성 낮음 (+ 주기성 : 근거 미확정) | 위와 같음 (임계값 이하) |
| LRDA | 편측성 높음 + 율동성 높음 | 위 2개 + delta_peak_sharp_max, rel_delta |
| GRDA | 편측성 낮음 + 율동성 높음 | 위와 같음 |
| Seizure | 시간 변화 큼 | center_ctx_power_absmax, center_ctx_centroid_absmax |
| Other | 판정 대상 아님 | - |

## 0. 설정 (목적 : 입력 탐색)

In [ ]:
import sys, os, json
from pathlib import Path

INPUT_ROOT = Path(os.environ.get("HMS_INPUT_ROOT", "/kaggle/input"))
def find_file(*patterns):
    for pat in patterns:
        hits = sorted(INPUT_ROOT.glob(pat)) if INPUT_ROOT.exists() else []
        if hits:
            return hits[0]
    return None

SRC_ROOT = Path(os.environ.get("HMS_SRC_DIR") or find_file("**/src/reasoning/rules.py").parents[2])
sys.path.insert(0, str(SRC_ROOT))
FEAT_PATH = Path(os.environ.get("HMS_FEATS") or find_file("**/profile/profile_features.csv"))
RECIPE_PATH = find_file("**/e8_recipe.json")
OOF = {m: Path(os.environ.get(f"HMS_{m}_OOF") or find_file(f"**/models/{m}/oof.csv")) for m in ["E4", "E5", "P1"]}

import numpy as np
import pandas as pd
from src.data.targets import PROB_COLS, CLASSES
from src.eval.metrics import evaluate
from src.reasoning.rules import AXES, FEATURES, fit_thresholds, check_claims
from src.reasoning.report import make_report, format_report, confidence

OUT_DIR = Path(os.environ.get("HMS_OUT_DIR", "/kaggle/working")) / "reasoning"
OUT_DIR.mkdir(parents=True, exist_ok=True)
RECIPE = json.load(open(RECIPE_PATH)) if RECIPE_PATH else {"models": ["E4", "E5"], "dl_weight": 0.85, "use_p1": True, "temperature": 1.0}
print("SRC_ROOT :", SRC_ROOT, "\nFEATS    :", FEAT_PATH, "\nRECIPE   :", RECIPE_PATH, RECIPE)
for m, p in OOF.items():
    print(f"{m} OOF  : {p}")

## 1. E8 확률 재구성 (목적 : 최종 제출 조합의 OOF 확률 / 판정에 쓰는 딥러닝 측 입력)

In [ ]:
feats = pd.read_csv(FEAT_PATH)
PRED = [f"pred_{c}" for c in CLASSES]
def oof(m):
    return pd.read_csv(OOF[m]).set_index("eeg_id").loc[feats.eeg_id, PRED].to_numpy()

dl = np.mean([oof(m) for m in RECIPE["models"]], axis=0)
w = RECIPE["dl_weight"]
prob = w * dl + (1 - w) * oof("P1") if RECIPE["use_p1"] else dl
if RECIPE.get("temperature", 1.0) != 1.0:
    z = np.log(np.clip(prob, 1e-7, 1)) / RECIPE["temperature"]; z = np.exp(z - z.max(1, keepdims=True)); prob = z / z.sum(1, keepdims=True)

m = evaluate(feats, prob)
print(f"E8 OOF 재구성 : 고득표 KL {m['kl_high']:.4f} (nb07 0.2955) | 편측 AUC {m['auc_lat']:.4f} | 주기성 AUC {m['auc_pd']:.4f} | "
      f"Seizure 민감도 {m['sens_at_spec90']:.4f} | {len(feats):,}건")

pred_cls = np.array(CLASSES)[prob.argmax(1)]
truth = feats.expert_consensus.str.lower().to_numpy()
Y = feats[PROB_COLS].to_numpy()
kl_row = (np.clip(Y, 1e-15, 1) * np.log(np.clip(Y, 1e-15, 1) / np.clip(prob, 1e-15, 1))).sum(1)
hv = feats.high_vote.to_numpy().astype(bool)
print(f"고득표 {hv.sum():,}건 | 최다 확률 클래스 = 전문가 다수 판정 일치율 {(pred_cls[hv] == truth[hv]).mean():.3f}")

## 2. 규칙 적용 (목적 : 판정마다 근거 지표 충족 여부 판정 / 누수 방지 : 임계값은 학습 fold에서만)

- fold k 구간에는 **fold k를 뺀 나머지 fold의 고득표 구간**에서 Youden 기준(민감도 + 특이도 최대)으로 정한 임계값을 적용
- 임계값이 fold 간에 크게 흔들리지 않는지(안정성)도 함께 확인

In [ ]:
res, thr_rows, THR_BY_FOLD = [], [], {}
for k in sorted(feats.fold.unique()):
    tr = feats[(feats.fold != k) & (feats.high_vote == 1)]
    thr = fit_thresholds(tr)
    THR_BY_FOLD[int(k)] = thr
    thr_rows.append({"fold": int(k), **{f: round(v, 4) for f, v in thr.items()}})
    va = feats.fold == k
    r = check_claims(feats[va], pred_cls[va.to_numpy()], thr)
    r["fold"] = int(k)
    res.append(r)
res = pd.concat(res).loc[feats.index]
thr_tab = pd.DataFrame(thr_rows).set_index("fold")
display(thr_tab)
cv = (thr_tab.std() / thr_tab.mean().abs()).round(3)
print("임계값 변동계수(fold 간 표준편차/평균) :", cv.to_dict(), "→", "안정" if cv.max() < 0.1 else "fold 간 변동 있음")
THR_ALL = fit_thresholds(feats[feats.high_vote == 1])    # 리포트·신규 구간용 (전체 학습 데이터 기준)

## 3. 평가 (목적 : 추론 로직 성능 목표 판정)

- **근거 일치율** : 고득표 구간 중 판정 대상(Other 제외)에서 '일치' 비율 → 목표 **80% 이상**
- **플래그 실효성** : '불일치'(전문의 우선 확인) 구간의 오답률이 '일치' 구간의 **2배 이상**이면 플래그가 오류를 잘 골라낸다고 판정
  - 오답 = 최다 확률 클래스 ≠ 전문가 다수 판정 (고득표 구간), 보조로 구간별 KL 평균도 비교

In [ ]:
df = feats[["eeg_id", "patient_id", "fold", "high_vote", "expert_consensus"]].copy()
df["pred"] = pred_cls
df["p_max"] = prob.max(1)
df["correct"] = pred_cls == truth
df["kl"] = kl_row
df = pd.concat([df, res.drop(columns="fold")], axis=1)

def summarize(d):
    chk = d[d.status != "판정 대상 아님"]
    con, inc = chk[chk.status == "일치"], chk[chk.status == "불일치"]
    err_c, err_i = 1 - con.correct.mean(), 1 - inc.correct.mean()
    return {"판정 대상": len(chk), "일치율": chk.status.eq("일치").mean(), "일치 n": len(con), "불일치 n": len(inc),
            "오답률(일치)": err_c, "오답률(불일치)": err_i, "오답률 비": err_i / max(err_c, 1e-9),
            "KL(일치)": con.kl.mean(), "KL(불일치)": inc.kl.mean(), "KL 비": inc.kl.mean() / max(con.kl.mean(), 1e-9)}

high, allv = summarize(df[df.high_vote == 1]), summarize(df)
summary = pd.DataFrame({"고득표": high, "전체(참고)": allv}).T
display(summary.round(3))

ok_rate, ok_flag = high["일치율"] >= 0.80, high["오답률 비"] >= 2.0
print(f"근거 일치율 {high['일치율']:.1%} (목표 ≥ 80%) → {'달성' if ok_rate else '미달'}")
print(f"플래그 실효성 : 오답률 불일치 {high['오답률(불일치)']:.1%} vs 일치 {high['오답률(일치)']:.1%} → {high['오답률 비']:.2f}배 "
      f"(목표 ≥ 2배) → {'달성' if ok_flag else '미달'} | KL 비 {high['KL 비']:.2f}배")

h = df[df.high_vote == 1]
hc = h[h.status != "판정 대상 아님"]
by_cls = pd.DataFrame({c: {"n": len(g), "일치율": g.status.eq("일치").mean(),
                           "정답률(일치)": g[g.status == "일치"].correct.mean(),
                           "정답률(불일치)": g[g.status == "불일치"].correct.mean()}
                       for c, g in hc.groupby("pred")}).T.round(3)
display(by_cls)
claim_cols = [f"claim_{a}" for a in AXES]
axis_tab = pd.DataFrame({AXES[a]["name"]: (h[h[f"claim_{a}"] >= 0][f"claim_{a}"].value_counts(normalize=True).sort_index().round(3))
                         for a in AXES}).rename(index={0: "근거 0개", 1: "1개", 2: "2개"})
print("축별 근거 충족 분포 (고득표, 해당 주장이 있는 구간)")
display(axis_tab)

## 3-2. 진단 분석 (목적 : 목표 미달 원인 파악 · 보완안 근거 / 1차 판정은 변경하지 않음)

- ① **오답 분해** : 고득표 오답을 혼동 유형별로 나누고, 유형마다 플래그가 붙은 비율 → 규칙이 확인할 수 있는 오답과 없는 오답 구분
- ② **축별 실효성** : 주장한 축이 실제로 틀렸는가 (예 : 편측 주장 → 전문가 판정이 전반) 를 근거 0개 vs 근거 있음으로 비교
- ③ **기준 민감도** : 지지 기준 1/2 vs 2/2, 임계값이 불안정한 지표(center_ctx_centroid_absmax) 제외
- ④ **확률 대조군** : 같은 수의 구간을 '최고 확률이 낮은 순'으로 플래그 / 확률 3분위 안에서 근거 불일치의 추가 효과 / 결합 플래그
- 모든 변형도 fold별 학습 fold 임계값(THR_BY_FOLD)으로 계산 → 누수 없음. 다만 변형 선택은 결과를 본 뒤 하므로 **사후 보완**으로 표기

In [ ]:
LAT, RDA, PDS = {"lpd", "lrda"}, {"lrda", "grda"}, {"lpd", "gpd"}
PAT = LAT | {"gpd", "grda"}

def confusion_type(t, p):
    if t == p:
        return "정답"
    if "other" in (t, p):
        return "Other 관련"
    if "seizure" in (t, p):
        return "Seizure 관련"
    lat_diff, rda_diff = (t in LAT) != (p in LAT), (t in RDA) != (p in RDA)
    if lat_diff and rda_diff:
        return "편측·율동 모두 다름"
    return "편측↔전반 (같은 패턴)" if lat_diff else "주기↔율동 (같은 위치)"

h = df[df.high_vote == 1].copy()
h["truth"] = h.expert_consensus.str.lower()
h["오답 유형"] = [confusion_type(t, p) for t, p in zip(h.truth, h.pred)]
err = h[~h.correct]
d1 = pd.DataFrame({"오답 n": err["오답 유형"].value_counts(),
                   "비율": err["오답 유형"].value_counts(normalize=True).round(3),
                   "플래그 비율": err.groupby("오답 유형").status.apply(lambda s: s.eq("불일치").mean()).round(3),
                   "판정 대상 아님 비율": err.groupby("오답 유형").status.apply(lambda s: s.eq("판정 대상 아님").mean()).round(3)})
print("① 고득표 오답 분해 (플래그 비율 = 그 유형 오답 중 '불일치'로 걸러진 비율)")
display(d1.sort_values("오답 n", ascending=False))
print(f"   참고 : 정답 구간의 플래그 비율 {h[h.correct].status.eq('불일치').mean():.3f} (오경보)")

rows2 = []
for ax, cls_hi, claim_cls in [("lateral", LAT, ["lpd", "lrda", "gpd", "grda"]), ("rhythmic", RDA, ["lrda", "grda"]),
                              ("seizure", {"seizure"}, ["seizure"])]:
    g = h[h.pred.isin(claim_cls) & (h[f"claim_{ax}"] >= 0)]
    if ax != "seizure":
        g = g[g.truth.isin(PAT)]                                   # 축 판단이 정의되는 패턴 클래스만
    high_claim = g.pred.isin(cls_hi) if ax != "lateral" else g.pred.isin(LAT)
    truth_hi = g.truth.isin(cls_hi) if ax != "lateral" else g.truth.isin(LAT)
    axis_wrong = high_claim != truth_hi
    zero = g[f"claim_{ax}"] == 0
    r0, r1 = axis_wrong[zero].mean(), axis_wrong[~zero].mean()
    rows2.append({"축": AXES[ax]["name"], "주장 n": len(g), "근거 0개 비율": round(zero.mean(), 3),
                  "축 오류율(근거 0개)": round(r0, 3), "축 오류율(근거 있음)": round(r1, 3), "비": round(r0 / r1, 2) if r1 > 0 else np.nan})
print("② 축별 실효성 (축 오류 = 주장한 축 방향이 전문가 판정과 다름)")
display(pd.DataFrame(rows2))

In [ ]:
import inspect
if "min_hits" not in inspect.signature(check_claims).parameters:
    # hms-src가 이전 버전이어도 진단이 돌도록 같은 동작의 함수를 노트북 안에 둠 (src/reasoning/rules.py 최신본과 동일)
    print("[알림] hms-src의 check_claims가 이전 버전 → 노트북 내 동일 함수로 진단 실행")
    from src.reasoning.rules import CLAIMS
    def check_claims(df_, pred_class, thr, min_hits=1, drop=()):
        pred_class = pd.Series(np.asarray(pred_class), index=df_.index).str.lower()
        out = pd.DataFrame(index=df_.index)
        for ax_key in AXES:
            out[f"claim_{ax_key}"] = -1
        ok_all, n_claims = pd.Series(True, index=df_.index), pd.Series(0, index=df_.index)
        for c, claims in CLAIMS.items():
            m = pred_class == c
            if not m.any():
                continue
            for ax_key, direction in claims:
                fs = [f for f in AXES[ax_key]["features"] if f not in drop]
                hits = sum(((df_.loc[m, f] > thr[f]) if direction == "high" else (df_.loc[m, f] <= thr[f])).astype(int) for f in fs)
                out.loc[m, f"claim_{ax_key}"] = hits
                ok_all.loc[m] &= hits >= min(min_hits, len(fs))
                n_claims.loc[m] += 1
        out["n_claims"] = n_claims
        out["status"] = np.where(n_claims == 0, "판정 대상 아님", np.where(ok_all, "일치", "불일치"))
        return out

def run_rules(min_hits=1, drop=()):
    out = pd.Series(index=feats.index, dtype=object)
    for k, thr in THR_BY_FOLD.items():
        va = (feats.fold == k).to_numpy()
        out[va] = check_claims(feats[va], pred_cls[va], thr, min_hits=min_hits, drop=drop).status.to_numpy()
    return out

def flag_stats(flag, mask_eval):
    f, c, k = flag[mask_eval], df.correct[mask_eval], df.kl[mask_eval]
    e1, e0 = 1 - c[f].mean(), 1 - c[~f].mean()
    return {"플래그 비율": f.mean(), "일치율": 1 - f.mean(), "오답률(플래그)": e1, "오답률(나머지)": e0,
            "오답률 비": e1 / max(e0, 1e-9), "KL 비": k[f].mean() / max(k[~f].mean(), 1e-9)}

hvm = df.high_vote.to_numpy().astype(bool)
variants = {"1차 : 1/2 (사전 규칙)": (1, ()), "2/2": (2, ()),
            "1/2, 불안정 지표 제외": (1, ("center_ctx_centroid_absmax",)), "2/2, 불안정 지표 제외": (2, ("center_ctx_centroid_absmax",))}
rows3, STATUS = [], {}
for name, (mh, dr) in variants.items():
    st = run_rules(mh, dr)
    STATUS[name] = st
    ev = hvm & (st != "판정 대상 아님").to_numpy()
    rows3.append({"변형": name, **flag_stats((st == "불일치").to_numpy(), ev)})

base_st = STATUS["1차 : 1/2 (사전 규칙)"]
ev = hvm & (base_st != "판정 대상 아님").to_numpy()
rule_flag = (base_st == "불일치").to_numpy()
n_flag = int(rule_flag[ev].sum())
pm = df.p_max.to_numpy()
low_p = np.zeros(len(df), bool)
low_p[np.where(ev)[0][np.argsort(pm[ev])[:n_flag]]] = True               # 같은 수만큼 최고 확률이 낮은 구간
rows3.append({"변형": f"대조 : 저확률 {n_flag}건 (근거 미사용)", **flag_stats(low_p, ev)})
q20 = np.quantile(pm[ev], 0.2)
comb = rule_flag | (pm < q20)
rows3.append({"변형": "결합 : 근거 불일치 또는 확률 하위 20%", **flag_stats(comb, ev)})
both = rule_flag & (pm < np.quantile(pm[ev], 0.5))
rows3.append({"변형": "결합 : 근거 불일치 그리고 확률 하위 50%", **flag_stats(both, ev)})
d3 = pd.DataFrame(rows3).set_index("변형").round(3)
print("③·④ 기준 민감도 · 확률 대조군 (고득표, 판정 대상 구간)")
display(d3)

tert = pd.qcut(pd.Series(pm[ev]).rank(method="first"), 3, labels=["확률 하위", "중위", "상위"])   # 동률 구간 경계 오류 방지
sub = df[ev].assign(tert=np.asarray(tert), flag=rule_flag[ev])
strat = pd.DataFrame({t: {"n": len(g), "오답률(불일치)": 1 - g[g.flag].correct.mean(), "오답률(일치)": 1 - g[~g.flag].correct.mean()}
                      for t, g in sub.groupby("tert", observed=True)}).T
strat["비"] = strat["오답률(불일치)"] / strat["오답률(일치)"].replace(0, np.nan)
print("④ 확률 3분위 안에서 근거 불일치의 추가 효과 (비 > 1 이면 확률 정보와 별개로 오류를 골라냄)")
display(strat.round(3))
d1.to_csv(OUT_DIR / "r1_diag_error_types.csv"); pd.DataFrame(rows2).to_csv(OUT_DIR / "r1_diag_axis.csv", index=False)
d3.to_csv(OUT_DIR / "r1_diag_variants.csv"); strat.to_csv(OUT_DIR / "r1_diag_prob_strata.csv")

## 3-3. 보완 규칙 R1b (사후 보완 1회, 10/26 진단 근거 / 1차 R1 결과는 공식 결과로 함께 보고)

- **근거 축 한정** : 판별력이 없는 시간 변화(Seizure) 축을 플래그에서 제외 (② : 1.00배) → Seizure 판정은 근거 문구만 표시
- **확신도 결합** : 최고 확률이 **학습 fold 고득표 구간의 하위 20%** 미만이면 플래그 (④ : 확률만으로 2.34배, 근거는 확신 구간에서 1.3배 추가 정보)
- **플래그 = 편측·율동 근거 불일치 또는 저확신**, 평가 대상은 **고득표 전체 구간**(Other 판정 포함 : 오답의 77%가 Other 관련이라 저확신 조건으로 일부 포착)
- 지표 2개 : 근거 일치율(편측·율동 주장이 있는 구간) / 플래그 오답률 비(고득표 전체)

In [ ]:
pred_b = np.where(pred_cls == "seizure", "other", pred_cls)        # Seizure 판정은 근거 점검 대상에서 제외
st_b = pd.Series(index=feats.index, dtype=object)
lowp = np.zeros(len(feats), bool)
q20_by_fold = {}
for k, thr in THR_BY_FOLD.items():
    va = (feats.fold == k).to_numpy()
    st_b[va] = check_claims(feats[va], pred_b[va], thr).status.to_numpy()
    q = float(np.quantile(df.p_max[(df.fold != k) & (df.high_vote == 1)], 0.2))   # 학습 fold로 저확신 기준
    q20_by_fold[int(k)] = round(q, 4)
    lowp[va] = df.p_max.to_numpy()[va] < q
flag_b = (st_b == "불일치").to_numpy() | lowp

hv_all = df.high_vote.to_numpy().astype(bool)
chk_b = hv_all & (st_b != "판정 대상 아님").to_numpy()
agree_b = float((st_b[chk_b] == "일치").mean())
s_b = flag_stats(flag_b, hv_all)
s_r1 = flag_stats(((base_st == "불일치").to_numpy()), hv_all)            # 비교 : 1차 규칙을 고득표 전체에 적용
s_p = flag_stats(lowp, hv_all)                                         # 비교 : 저확신 조건만
cmp_b = pd.DataFrame({"R1 1차 (근거만, 3축)": s_r1, "저확신만 (하위 20%)": s_p, "R1b (편측·율동 근거 또는 저확신)": s_b}).T.round(3)
print("저확신 기준(fold별 최고 확률 하위 20%) :", q20_by_fold)
display(cmp_b)
ok_rate_b, ok_flag_b = agree_b >= 0.80, s_b["오답률 비"] >= 2.0
print(f"R1b 근거 일치율(편측·율동 주장 구간) {agree_b:.1%} → {'달성' if ok_rate_b else '미달'} (목표 ≥ 80%)")
print(f"R1b 플래그 오답률 비 {s_b['오답률 비']:.2f}배 (플래그 {s_b['플래그 비율']:.1%}, 오답률 {s_b['오답률(플래그)']:.1%} vs {s_b['오답률(나머지)']:.1%}) "
      f"→ {'달성' if ok_flag_b else '미달'} (목표 ≥ 2배)")
add_b = s_b["오답률 비"] - s_p["오답률 비"]
print(f"근거의 추가 기여 : R1b − 저확신만 = {add_b:+.2f}배 ({'근거가 확률 정보에 더해 오류를 추가로 골라냄' if add_b > 0 else '추가 기여 없음'})")
cmp_b.to_csv(OUT_DIR / "r1b_comparison.csv")
df["status_r1b"], df["flag_r1b"] = st_b.to_numpy(), flag_b

## 4. 판독 보조 리포트 + 사례 리포트 8건 (목적 : '소견 + 판정 + 근거 + 신뢰도' 출력, M5 사례 리포트 재료)

- 리포트 규칙 (22차 113~114) : **플래그 = 저확신**(최고 확률 < 학습 fold 고득표 하위 20%), 근거 불일치는 **'어느 판단을 재확인할지'** 보조 문구, 시간 변화 근거는 참고 표시
- **사례 선정 규칙 (사전 지정, 고득표 구간)**
  1~5. 클래스별(Seizure·LPD·GPD·LRDA·GRDA) 정답·근거 일치 중 최고 확률 1건
  6. 저확신 플래그로 걸러진 오답 (Other 경계) 중 KL 최대 1건
  7. 저확신 플래그는 없지만 편측 재확인 문구가 붙은 오답 중 최고 확률 1건 : 근거가 의심 축을 알려준 사례
  8. 플래그·재확인 문구 모두 없던 오답 중 최고 확률 1건 : 시스템이 놓친 사례 (한계)
- 각 사례 : 리포트 문구 + 몽타주 파형 20초(15~35초, 판정 대상 중앙 10초 음영) + 10분 스펙트로그램 그림 → `reasoning/case_reports/` (case_reports.md, PNG)

In [ ]:
import inspect, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
assert "p_low" in inspect.signature(make_report).parameters, (
    "hms-src가 이전 버전입니다 : Datasets → hms-src → New Version으로 최신 hms-src.zip을 올리고, "
    "nb06 Input의 hms-src를 최신 버전으로 바꾼 뒤 Run → Restart 후 다시 실행하세요.")
from src.data.montage import CHANNEL_NAMES

P_LOW = float(np.mean(list(q20_by_fold.values())))          # 리포트용 저확신 기준 (fold별 하위 20%의 평균)
PROC = Path(os.environ.get("HMS_PROC_DIR") or find_file("**/hms_processed/eeg50.npy").parent)
meta_p = pd.read_csv(PROC / "meta.csv")
assert (meta_p.eeg_id.to_numpy() == feats.eeg_id.to_numpy()).all(), "profile_features와 hms_processed의 구간 순서가 다릅니다"
EEG = np.load(PROC / "eeg50.npy", mmap_mode="r")
SPEC = np.load(PROC / "spec10.npy", mmap_mode="r")
CASE_DIR = OUT_DIR / "case_reports"
CASE_DIR.mkdir(exist_ok=True)

h = df[df.high_vote == 1].copy()
h["truth"] = h.expert_consensus.str.lower()
h["low_conf"] = h.p_max < P_LOW
recheck_lat = (h.claim_lateral == 0) & h.pred.isin(["lpd", "gpd", "lrda", "grda"])
recheck_rda = (h.claim_rhythmic == 0) & h.pred.isin(["lrda", "grda"])
h["recheck"] = recheck_lat | recheck_rda

picks = []
for c in ["seizure", "lpd", "gpd", "lrda", "grda"]:
    g = h[(h.pred == c) & h.correct & ~h.low_conf & ~h.recheck]
    if len(g):
        picks.append(("정답·근거 일치", g.sort_values("p_max", ascending=False).index[0]))
g = h[~h.correct & h.low_conf & ((h.truth == "other") | (h.pred == "other"))]
if len(g):
    picks.append(("저확신 플래그로 걸러진 오답 (Other 경계)", g.sort_values("kl", ascending=False).index[0]))
g = h[~h.correct & ~h.low_conf & recheck_lat & (h.truth.isin(["lpd", "lrda"]) != h.pred.isin(["lpd", "lrda"]))]
if len(g):
    picks.append(("저확신 아닌 오답 + 편측 재확인 문구 (근거가 의심 축을 알려줌)", g.sort_values("p_max", ascending=False).index[0]))
g = h[~h.correct & ~h.low_conf & ~h.recheck]
if len(g):
    picks.append(("플래그·재확인 없이 놓친 오답 (한계 사례)", g.sort_values("p_max", ascending=False).index[0]))

def plot_case(i, title, path):
    x = np.asarray(EEG[i], dtype=np.float32)
    s = np.asarray(SPEC[i], dtype=np.float32)
    fig = plt.figure(figsize=(15, 6.2))
    ax = fig.add_axes([0.05, 0.08, 0.55, 0.82])
    seg = slice(15 * 200, 35 * 200)                         # 판독 화면처럼 20초만 표시 (15~35초, 중앙 10초 포함)
    t = np.arange(x.shape[1])[seg] / 200
    scale = np.percentile(np.abs(x[:, seg]), 99) + 1e-6
    ypos = [-(ch * 1.2 + (ch // 4) * 0.8) for ch in range(16)]   # 체인(LL·RL·LP·RP) 사이 간격 : 판독 화면 배치
    for ch in range(16):
        ax.plot(t, x[ch, seg] / scale + ypos[ch], lw=0.6, color="#1f3b57")
    ax.axvspan(20, 30, color="#f2a541", alpha=0.18, lw=0)
    bar = 100 / scale                                                 # 100 µV 눈금 막대 (같은 배율의 모든 채널 공통)
    ax.plot([34.6, 34.6], [ypos[-1] - 1.0, ypos[-1] - 1.0 + bar], color="black", lw=1.5)
    ax.text(34.45, ypos[-1] - 1.0 + bar / 2, "100 µV", ha="right", va="center", fontsize=7)
    ax.set_yticks(ypos); ax.set_yticklabels(CHANNEL_NAMES, fontsize=7)
    ax.set_xlabel("time in 50-s segment (s) | shaded = central 10 s (expert-labelled window)"); ax.set_xlim(15, 35)
    ax.set_title(title, fontsize=10, loc="left")
    for r, name in enumerate(["LL", "RL", "LP", "RP"]):
        a2 = fig.add_axes([0.66, 0.08 + (3 - r) * 0.21, 0.32, 0.18])
        a2.imshow(s[r], aspect="auto", origin="lower", cmap="magma", extent=[-300, 300, 0.5, 20])
        a2.axvline(0, color="white", lw=0.6, ls="--")
        a2.set_ylabel(f"{name}\nHz", fontsize=7); a2.tick_params(labelsize=6)
        if r < 3:
            a2.set_xticklabels([])
    a2.set_xlabel("time from window centre (s) | log power", fontsize=7)
    fig.savefig(path, dpi=110); plt.close(fig)

md_lines = ["# 판독 보조 리포트 사례 (HMS EEG, 고득표 구간)", "",
            f"- 모델 : E8 (0.85·mean(E4, E5) + 0.15·P1v2), 근거 임계값 : 학습 데이터 고득표 구간 Youden, 저확신 기준 {P_LOW:.2f}",
            "- 연구·학습 목적 예시이며 임상 진단 도구가 아님", ""]
for n, (kind, i) in enumerate(picks, 1):
    r = df.loc[i]
    rep = make_report(feats.loc[i], prob[i], THR_ALL, p_low=P_LOW)
    votes = ", ".join(f"{c} {v:.0%}" for c, v in zip(["Seizure", "LPD", "GPD", "LRDA", "GRDA", "Other"], feats.loc[i, PROB_COLS]) if v > 0)
    img = f"case{n}_{int(r.eeg_id)}.png"
    plot_case(i, f"Case {n} | eeg_id {int(r.eeg_id)} | expert {r.expert_consensus} | model {r.pred.upper()} {r.p_max:.2f}", CASE_DIR / img)
    md_lines += [f"## 사례 {n}. {kind}", "",
                 f"- eeg_id {int(r.eeg_id)} | 전문가 투표 분포 : {votes} | 모델 판정 : {'정답' if r.correct else '오답'}", "",
                 "```", format_report(rep), "```", "", f"![case {n}]({img})", ""]
    print(f"── 사례 {n} : {kind} | eeg_id {int(r.eeg_id)} | 전문가 {r.expert_consensus} (투표 {votes})")
    print(format_report(rep), "\n")
(CASE_DIR / "case_reports.md").write_text("\n".join(md_lines), encoding="utf-8")
cand = pd.DataFrame([{"사례": n, "유형": k, "eeg_id": int(df.loc[i, "eeg_id"]), "전문가": df.loc[i, "expert_consensus"],
                      "판정": df.loc[i, "pred"], "p_max": round(df.loc[i, "p_max"], 3), "정답": bool(df.loc[i, "correct"])}
                     for n, (k, i) in enumerate(picks, 1)])
display(cand)
print("저장 :", sorted(p.name for p in CASE_DIR.iterdir()))

## 5. 저장 (목적 : 결과 기록 / 출력 : r1_samples.csv, r1_summary.json, case_candidates.csv, results_row_r1.csv)

In [ ]:
df.to_csv(OUT_DIR / "r1_samples.csv", index=False)
thr_tab.to_csv(OUT_DIR / "r1_thresholds_by_fold.csv")
by_cls.to_csv(OUT_DIR / "r1_by_class.csv")
cand.to_csv(OUT_DIR / "case_candidates.csv", index=False)
with open(OUT_DIR / "r1_summary.json", "w") as f:
    json.dump({"high_vote": {k: float(v) for k, v in high.items()}, "all": {k: float(v) for k, v in allv.items()},
               "thresholds_all": THR_ALL, "recipe": RECIPE, "features": FEATURES,
               "target_rate_met": bool(ok_rate), "target_flag_met": bool(ok_flag),
               "r1b": {"agree_rate": agree_b, **{k: float(v) for k, v in s_b.items()}, "low_conf_threshold_by_fold": q20_by_fold,
                       "target_rate_met": bool(ok_rate_b), "target_flag_met": bool(ok_flag_b)}}, f, indent=2, ensure_ascii=False)
record = {"exp": "R1", "hypothesis": "reasoning consistency (goal 2)",
          "change": "E8 prob + H5-supported evidence (lateral 2, rhythmic 2, seizure-ctx 2), thresholds fit on train folds (Youden)",
          "cv_kl_all": None, "cv_kl_high": None, "public_kl": None, "private_kl": None, "auc_lat": None, "auc_pd": None,
          "sens_at_spec90": None, "gpu_hours": 0, "kaggle_version": None,
          "verdict": f"consistency {high['일치율']:.1%} (target 80% {'met' if ok_rate else 'not met'}); "
                     f"flag error ratio {high['오답률 비']:.2f}x (target 2x {'met' if ok_flag else 'not met'})",
          "note": f"high-vote checkable {high['판정 대상']}, error consistent {high['오답률(일치)']:.3f} / inconsistent {high['오답률(불일치)']:.3f}, KL ratio {high['KL 비']:.2f}"}
record_b = dict(record, exp="R1b", hypothesis="reasoning consistency (post-hoc revision 1)",
                change="lateral+rhythmic evidence only (seizure axis removed) OR low confidence (p_max < train-fold 20th pct); eval on all high-vote",
                verdict=f"evidence agreement {agree_b:.1%} ({'met' if ok_rate_b else 'not met'}); flag error ratio {s_b['오답률 비']:.2f}x ({'met' if ok_flag_b else 'not met'})",
                note=f"flag share {s_b['플래그 비율']:.3f}; low-conf only {s_p['오답률 비']:.2f}x; evidence add {add_b:+.2f}x")
pd.DataFrame([record, record_b]).to_csv(OUT_DIR / "results_row_r1.csv", index=False)
print("저장 파일 :", sorted(p.name for p in OUT_DIR.iterdir()))

## 다음 단계
- 공유할 결과 : 섹션 1 재구성 줄, 섹션 2 임계값 안정성 줄, 섹션 3 **요약표·판정 2줄·클래스별 표·축별 분포**, 섹션 4 리포트 예시
- 3-2 진단표 4개(오답 분해, 축별, 변형 비교, 확률 3분위)를 공유 → 보완안 1개 확정
- 목표 미달 시 : 규칙 1회 보완 (예 : 근거 2개 모두 충족 요구 / 축별 임계값 조정) 후 재평가, 보완 내용은 코드북에 기록